# step5 통제 — 지침 전이의 거품을 걷어낸다 (RQ3)

**어느 스텝·어느 RQ:** step5(지침 인과, RQ3)에 **빠져 있던 통제 조건**을 채운다.

**왜 필요한가.**
step5는 지침의 지시어 단어를 반대 지침 것으로 덮어 행동이 넘어가는지 쟀고, 전이율 0.42~0.89를 얻었다.
그런데 **통제 조건이 하나도 없었다.**

step3(코드)에서 같은 걸 확인해 보니, **아무 값이나 덮어도 모델은 흔들렸고 그 몫이 회복률의 절반을
넘었다.** 즉 지금 step5의 숫자는 "지침 정보가 옮겨간 몫"과 "그냥 덮어서 흔들린 몫"이 **섞여 있다.**
지금 데이터로는 둘을 못 가른다.

**무엇을 추가하나.** 덮어넣을 값의 출처를 두 가지 더 만든다. 프롬프트도 층도 그대로다.

| 공여 | 무엇을 덮나 | 정상이라면 |
|---|---|---|
| `opposite` (기존) | 반대 지침의 지시어 | 크게 넘어감 |
| **`self`** (자기 통제) | **같은 지침의 같은 지시어** | **≈ 0** — 새 정보가 하나도 안 들어가고 '덮는 행위'만 남는다 |
| **`unrelated_word`** (음성 통제) | 같은 지침의 무관한 단어(`project`) | **≈ 0** — 표기와 무관한 내용을 덮는다 |

**결과가 어느 쪽으로 나오든 무슨 뜻인지 미리 밝힌다.**

| 나오는 그림 | 뜻 |
|---|---|
| 두 통제 ≈ 0 | 기존 전이율이 **거의 다 진짜**다. 지금 수치를 그대로 쓸 수 있다 |
| 두 통제가 큼 | 기존 수치에 거품이 꼈다. **처치 − 통제**로 다시 보고한다. 크기는 줄지만 **"내용 쪽에 실린다"는 결론 자체는 유지**된다(어텐션 쪽은 어차피 통제와 같은 값이므로) |

**부하.** 전 층 스윕이 아니다. **봉우리 층 부근만** 돈다(모델별로 이미 안다).
조건 수는 모델당 42묶음 × 방향 2 × 통제 2 = 168개인데 층이 몇 개뿐이라 step5 본실험보다 훨씬 가볍다.

**모델 하나씩.** 셀 ④ `PICK`을 바꿔 네 번. 끊겨도 저장된 건 건너뛴다.

In [ ]:
# ② 환경 — 설치, GPU 확인, 무작위값 42 고정
!pip install -q -r requirements.txt

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('무작위값 고정:', SEED)


In [ ]:
# ③ 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
BRANCH = 'integration/step1-5'
!git fetch --quiet origin $BRANCH
!git checkout $BRANCH
!git pull --quiet origin $BRANCH
!pip install -e . -q
import sys; sys.path.insert(0, 'src')
print('브랜치:', BRANCH)


In [ ]:
# ④ 조건 설정 — 모델 하나, 봉우리 층 부근, 통제 공여 2종
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]
# step5 본실험에서 확인된 봉우리 층 (docs/step5/결과_재집계.md)
PEAK = {'qwen': 27, 'deepseek': 17, 'llama': 24, 'stability': 19}

# ★ 이번에 돌릴 모델 하나 (0=qwen, 1=deepseek, 2=llama, 3=stable)
PICK = 0
MODEL = MODELS[PICK]
LAYER = PEAK[MODEL.family]          # 봉우리 층 하나만 — 층마다 조건을 따로 만든다
print('이번 모델:', MODEL.family, '| 볼 층: L', LAYER)

BLOCKS = list(range(42))
DONORS = ['self', 'unrelated_word']        # 통제 2종 (처치 opposite는 이미 돌렸다)

conditions = []
for block in BLOCKS:
    pre = PrecedingCode(n_compliant=6, n_functions=12, composition=Composition.POOL, pool_block=block)
    for notation in (Notation.CAMEL, Notation.SNAKE):
        for donor in DONORS:
            conditions.append(Condition(
                model=MODEL, preceding=pre,
                instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=notation),
                intervention=Intervention(kind=InterventionKind.KEY_VALUE, layers=[LAYER],
                                          donor=donor, target='instruction',
                                          # 세 방식을 한 번에 잰다 — 본실험(내용만)과 방식을 맞춰야
                                          # 거품을 뺄 수 있다. 프롬프트는 한 번만 통과하므로 비용은 작다.
                                          kinds=('key', 'value', 'key_value')),
                seed=SEED, token_unit='mean'))
print('조건 수:', len(conditions), f'(묶음 {len(BLOCKS)} × 방향 2 × 통제 {len(DONORS)}), 각 조건마다 방식 3가지')


In [ ]:
# ⑤ 실행 — 층 목록을 돌며 조건마다 즉시 저장(재개)
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
import numpy as np

STEP = 'step5_instr-cause-control'
todo = [c for c in conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 전체 {len(conditions)} / 남은 {len(todo)}')

if todo:
    handle = load_model(MODEL)
    print(f'  층수 {handle.num_layers}')
    seen = []
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle)          # 개입 — 조건이 지정한 층 하나
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ3',
                                 prediction='통제이므로 전이율이 0 근처여야 정상'))
        ex = out.metrics.extra
        v = out.metrics.per_layer[LAYER].get('value__recovery')
        if v is not None: seen.append(v)
        if i % 10 == 0 or i == len(todo):
            print(f'  [{i}/{len(todo)}] 공여={ex["donor"]:<24} '
                  f'지금까지 넘어감(내용만) 평균 {np.mean(seen):.3f} (0에 가까워야 정상)')
    del handle
    import torch, gc; gc.collect(); torch.cuda.empty_cache()
print('완료')


In [ ]:
# ⑥ 결과 로드
from harness import result_path
from harness.results import load_result
recs = [load_result(result_path(c, step='step5_instr-cause-control')) for c in conditions
        if result_path(c, step='step5_instr-cause-control').exists()]
print('불러온 조건:', len(recs))


In [ ]:
# ⑦ 요약 — 방식별로 통제가 얼마인지, 그래서 순효과가 얼마인지
import numpy as np, glob, json as _json
from collections import defaultdict

KINDS = ['value', 'key', 'key_value']
KO = {'value': '내용만', 'key': '어텐션만', 'key_value': '둘다'}

# (1) 통제 — 이번에 돌린 것
ctrl = defaultdict(lambda: defaultdict(list))
n_und = 0
for r in recs:
    ex = r.metrics.extra
    if ex.get('undecidable'):
        n_und += 1; continue
    vals = r.metrics.per_layer.get(LAYER) or r.metrics.per_layer.get(str(LAYER)) or {}
    for k in KINDS:
        x = vals.get(f'{k}__recovery')
        if x is not None:
            ctrl[ex['donor']][k].append(x)

# (2) 처치 — step5 본실험에서 같은 모델·같은 층을 읽어 온다
treat = defaultdict(list)
for f in glob.glob('results/step5_instr-cause/*.json'):
    d = _json.load(open(f))
    if d['condition']['model']['family'] != MODEL.family:
        continue
    if d['metrics']['extra'].get('undecidable'):
        continue
    vals = d['metrics']['per_layer'].get(str(LAYER)) or d['metrics']['per_layer'].get(LAYER) or {}
    for k in KINDS:
        x = vals.get(f'{k}__recovery')
        if x is not None:
            treat[k].append(x)

print(f'[{MODEL.family}] 봉우리 층 L{LAYER} | 판정 불가로 뺀 조건 {n_und}개')
print()
print(f"{'방식':<10}{'처치(반대지침)':>14}{'자기통제':>11}{'무관단어':>11}"
      f"{'순효과(자기)':>13}{'순효과(무관)':>13}")
for k in KINDS:
    t = np.mean(treat[k]) if treat[k] else float('nan')
    s = np.mean(ctrl['control_self'][k]) if ctrl['control_self'][k] else float('nan')
    u = np.mean(ctrl['control_unrelated_word'][k]) if ctrl['control_unrelated_word'][k] else float('nan')
    print(f'{KO[k]:<10}{t:>14.3f}{s:>11.3f}{u:>11.3f}{t-s:>13.3f}{t-u:>13.3f}')

print()
print('읽는 법:')
print('  · 처치와 통제를 **같은 방식끼리** 비교해야 한다(내용만은 내용만끼리).')
print('  · 순효과 = 처치 − 통제. 이 값이 논문에 실을 숫자다.')
print('  · 내용만의 순효과가 어텐션만의 순효과보다 크게 남으면 결론 유지.')
print('  · 통제 두 종류가 비슷하면 해석이 깔끔하다. 많이 다르면 그 사실을 적어야 한다.')


In [ ]:
# ⑧ 결과 폴더 zip 다운로드
import shutil
shutil.make_archive('step5_control_results', 'zip', 'results/step5_instr-cause-control')
try:
    from google.colab import files; files.download('step5_control_results.zip')
except Exception as e:
    print('Colab 아님(로컬):', e)
